### Pozivanje LLM modela

In [ ]:
import pandas as pd
import joblib

df = pd.read_csv("../data/processed/final.csv")
model = joblib.load("../models/final_model.pkl")
threshold = joblib.load("../models/threshold.pkl")

In [2]:
df.columns.to_list()

['code_module',
 'code_presentation',
 'id_student',
 'gender',
 'region',
 'highest_education',
 'imd_band',
 'age_band',
 'num_of_prev_attempts',
 'studied_credits',
 'disability',
 'final_result',
 'avg_days_before_deadline_CMA',
 'avg_days_before_deadline_TMA',
 'avg_score_CMA',
 'avg_score_TMA',
 'count_submitted_CMA',
 'count_submitted_TMA',
 'std_score_CMA',
 'std_score_TMA',
 'total_assessments_CMA',
 'total_assessments_TMA',
 'submission_rate_TMA',
 'submission_rate_CMA',
 'total_clicks',
 'last_active_day',
 'unique_activities']

In [3]:
df["target"] = df["final_result"].isin(["Fail", "Withdrawn"]).astype(int)
df["year"] = df["code_presentation"].astype(str).str[:4]

test_df = df[df["year"] == "2014"].copy()
X_test = test_df.drop(columns=["final_result", "target", "id_student", "year"],errors="ignore")
y_test = test_df["target"]

In [4]:
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= threshold).astype(int)

test_df["risk_probability"] = y_proba
test_df["prediction"] = y_pred

rizicni_studenti = test_df[test_df["prediction"] == 1].copy()
rizicni_studenti[["id_student", "code_module","code_presentation","final_result","risk_probability"]].head()

,id_student,code_module,code_presentation,final_result,risk_probability
359,24734,AAA,2014J,Pass,0.831003
370,65002,AAA,2014J,Fail,0.500507
374,85092,AAA,2014J,Pass,0.762018
377,87431,AAA,2014J,Withdrawn,0.743483
379,95755,AAA,2014J,Withdrawn,0.871757


In [5]:
student = rizicni_studenti.iloc[0]
student

code_module                                    AAA
code_presentation                            2014J
id_student                                   24734
gender                                           F
region                                South Region
highest_education               Lower Than A Level
imd_band                                    60-70%
age_band                                      0-35
num_of_prev_attempts                             0
studied_credits                                 60
disability                                       N
final_result                                  Pass
avg_days_before_deadline_CMA                   0.0
avg_days_before_deadline_TMA                  -7.0
avg_score_CMA                                  0.0
avg_score_TMA                                 39.5
count_submitted_CMA                            0.0
count_submitted_TMA                            2.0
std_score_CMA                                  0.0
std_score_TMA                  

In [6]:
prompt = f"""
Student je označen kao rizičan na osnovu prethodno treniranog modela.

Rezultat modela:
- Verovatnoća rizika: {round(student["risk_probability"], 4)}
- Prag odlučivanja: {threshold}

Osnovne informacije:
- Kurs: {student["code_module"]}
- Prezentacija kursa: {student["code_presentation"]}

Rezultati na zadacima:
- Prosečna CMA ocena: {student["avg_score_CMA"]}
- Prosečna TMA ocena: {student["avg_score_TMA"]}
- Standardna devijacija CMA ocena: {student["std_score_CMA"]}
- Standardna devijacija TMA ocena: {student["std_score_TMA"]}

Predaja zadataka:
- Broj predatih CMA zadataka: {student["count_submitted_CMA"]}
- Ukupan broj CMA zadataka: {student["total_assessments_CMA"]}
- Stopa predaje CMA zadataka: {student["submission_rate_CMA"]}
- Broj predatih TMA zadataka: {student["count_submitted_TMA"]}
- Ukupan broj TMA zadataka: {student["total_assessments_TMA"]}
- Stopa predaje TMA zadataka: {student["submission_rate_TMA"]}

Odnos prema rokovima:
- Prosečan broj dana pre roka za CMA: {student["avg_days_before_deadline_CMA"]}
- Prosečan broj dana pre roka za TMA: {student["avg_days_before_deadline_TMA"]}

Aktivnost na VLE platformi:
- Ukupan broj klikova: {student["total_clicks"]}
- Poslednji dan aktivnosti: {student["last_active_day"]}
- Broj različitih aktivnosti: {student["unique_activities"]}

Zadatak:
Napiši kratko objašnjenje zašto je student označen kao rizičan i predloži 2-3 mere podrške.

Pravila:
- Odgovor napiši na srpskom jeziku.
- Odgovor neka bude kratak, jasan i neutralan.
- Ne koristi demografske ili osetljive atribute kao razlog rizika.
- Ne tvrdi da je neka vrednost visoka ili niska ako to nije jasno iz konteksta.
- Ako je neka vrednost 0, napiši neutralno, npr. "nema evidentiranih CMA predaja".
- Nemoj donositi konačnu odluku o studentu.
"""

print(prompt)


Student je označen kao rizičan na osnovu prethodno treniranog modela.

Rezultat modela:
- Verovatnoća rizika: 0.831
- Prag odlučivanja: 0.4

Osnovne informacije:
- Kurs: AAA
- Prezentacija kursa: 2014J

Rezultati na zadacima:
- Prosečna CMA ocena: 0.0
- Prosečna TMA ocena: 39.5
- Standardna devijacija CMA ocena: 0.0
- Standardna devijacija TMA ocena: 2.1213203435596424

Predaja zadataka:
- Broj predatih CMA zadataka: 0.0
- Ukupan broj CMA zadataka: 0.0
- Stopa predaje CMA zadataka: 0.0
- Broj predatih TMA zadataka: 2.0
- Ukupan broj TMA zadataka: 2.0
- Stopa predaje TMA zadataka: 1.0

Odnos prema rokovima:
- Prosečan broj dana pre roka za CMA: 0.0
- Prosečan broj dana pre roka za TMA: -7.0

Aktivnost na VLE platformi:
- Ukupan broj klikova: 189.0
- Poslednji dan aktivnosti: 86.0
- Broj različitih aktivnosti: 6.0

Zadatak:
Napiši kratko objašnjenje zašto je student označen kao rizičan i predloži 2-3 mere podrške.

Pravila:
- Odgovor napiši na srpskom jeziku.
- Odgovor neka bude kratak,

In [7]:
from openai import OpenAI

openai_client = OpenAI()
openai_response = openai_client.responses.create(model="gpt-4o", input=prompt)
openai_odgovor = openai_response.output_text
print(openai_odgovor)

Student je označen kao rizičan jer model procenjuje verovatnoću rizika od 0.831, što je iznad postavljenog praga od 0.4. Ključne indikacije uključuju prosečnu TMA ocenu od 39.5 i kašnjenje u predaji TMA zadataka za prosečno 7 dana. Nema evidentiranih predaja ni ocena za CMA zadatke, a zabeležena je određena aktivnost na VLE platformi sa 189 klikova.

Predložene mere podrške:
1. **Akademska podrška**: Organizovati konsultacije sa mentorom ili nastavnikom kako bi se identifikovali problemi u razumevanju gradiva i prilagodio plan učenja.
   
2. **Upravljanje vremenom**: Ponuditi radionice ili resurse o veštinama upravljanja vremenom kako bi se poboljšalo planiranje i izvršavanje zadataka pre roka.

3. **Motivaciona podrška**: Obezbediti motivacione sesije ili podršku kroz studijske grupe kako bi se povećala angažovanost i interesovanje za kurs.


In [8]:
openai_mini_response = openai_client.responses.create(model="gpt-4o-mini",input=prompt)
openai_mini_odgovor = openai_mini_response.output_text
print(openai_mini_odgovor)

Student je označen kao rizičan zbog visoke verovatnoće rizika od 0.831, što je znatno iznad postavljenog praga odlučivanja. Rezultati pokazuju da student nema evidentiranih CMA predaja, dok je stopa predaje TMA zadataka 1.0, ali je prosečan broj dana pre roka za TMA negativan (-7.0), što sugeriše kašnjenje.

Predložene mere podrške:

1. **Individualne konsultacije**: Organizovati sastanak sa studentom kako bi se razgovaralo o izazovima sa rokovima i strategijama za poboljšanje.

2. **Pružanje dodatnih resursa**: Osigurati pristup materijalima i pomoćnim alatima koji mogu pomoći studentu u razumevanju i izvršavanju zadataka.

3. **Mentorski program**: Uključiti studenta u mentorske aktivnosti ili grupe za učenje kako bi imao podršku od kolega i nastavnika.


In [9]:
from google import genai

gemini_client = genai.Client()
gemini_response = gemini_client.models.generate_content(model="gemini-3.6-flash",contents=prompt)
gemini_odgovor = gemini_response.text

print(gemini_odgovor)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


**Objašnjenje rizika:**

Model je identifikovao studenta kao rizičnog (verovatnoća rizika iznosi 0.831, što je iznad praga odlučivanja od 0.4) na osnovu sledećih pokazatelja:

* **Rezultati na zadacima:** Prosečna ocena na predatim TMA zadacima je 39.5.
* **Poštovanje rokova:** TMA zadaci su u proseku predavani 7 dana nakon definisanog roka (-7.0 dana).
* **Aktivnost na platformi:** Zabeleženo je ukupno 189 klikova u okviru 6 različitih tipova aktivnosti, a poslednji zabeleženi pristup VLE platformi bio je 86. dana.
* **CMA obaveze:** Nema evidentiranih CMA zadataka niti predaja.

---

**Predložene mere podrške:**

1. **Akademske konsultacije:** Organizovati razgovor sa tutorom predmeta radi pružanja dodatnih pojašnjenja gradiva i podrške pri izradi narednih TMA zadataka.
2. **Podrška u planiranju i upravljanju vremenom:** Ponuditi radionicu ili savetovanje o organizaciji učenja kako bi se predupredilo kašnjenje sa predajom obaveza.
3. **Provera statusa i ponovno uključivanje:** Kontak

In [10]:
gemini_lite_response = gemini_client.models.generate_content(model="gemini-3.5-flash-lite",contents=prompt)
gemini_lite_odgovor = gemini_lite_response.text
print(gemini_lite_odgovor)

Na osnovu analize podataka, student je označen kao rizičan jer model procenjuje verovatnoću rizika na 0.831, što je iznad praga odlučivanja od 0.4. Glavni pokazatelji koji doprinose ovoj proceni su niska prosečna TMA ocena (39.5), kašnjenje u predaji TMA zadataka (prosečno 7 dana nakon roka) i prestanak aktivnosti na VLE platformi nakon 86. dana sa ukupno 189 klikova. Takođe, nema evidentiranih CMA predaja.

**Predlog mera podrške:**
1. **Akademsko savetovanje:** Organizovati razgovor sa studentom radi identifikovanja poteškoća u razumevanju gradiva koje utiču na TMA ocene i probijanje rokova.
2. **Podrška u organizaciji vremena:** Uključiti studenta u savetovalište za upravljanje vremenom i planiranje obaveza u vezi sa rokovima zadataka.
3. **Praćenje VLE aktivnosti:** Ponuditi tehničku i mentorsku podršku radi obnavljanja angažovanja i redovnog praćenja materijala na platformi.
